## Hedonic Price Modeling of Housing Data  

### 1. Introduction

The purpose of this project is to model and predict house prices using a hedonic regression framework. Housing prices are determined by a bundle of structural and neighborhood characteristics such as lot size, number of rooms, location, and amenities. Because these relationships are often nonlinear, careful attention must be paid to functional form, interaction effects, endogeneity, and predictive performance.

Using data from Anglin and Gencay (1996), we estimate several econometric models, perform specification tests, and evaluate out-of-sample prediction accuracy. The main goals of the analysis are:

- To test for linearity and appropriate functional form  
- To determine whether logarithmic transformations improve model fit  
- To analyze interaction effects between lot size and other characteristics  
- To examine potential endogeneity  
- To evaluate the predictive power of the final model  

All hypothesis tests are performed at the 5% significance level.

### 2. Data Description

The dataset contains **546 observations** of housing transactions. The dependent variable is the sale price of the house, while the explanatory variables describe physical features and location.

The average house sells for **68,122**, with substantial variation (std. dev. = 26,703).  
Average lot size is **5,150 sq. ft.**, and houses have about **3 bedrooms**, **1.3 bathrooms**, and **1.8 stories**.  
Most houses have a driveway, while fewer have recreational rooms, finished basements, or central air conditioning.

### 3. Methodology

We estimate several OLS regression models and evaluate their specification using the **Ramsey RESET test**. Logarithmic transformations are applied to improve functional form. Interaction effects are examined and tested jointly using F-tests. A general-to-specific procedure is used to eliminate insignificant interactions.

Finally, the predictive performance of the model is evaluated using **out-of-sample prediction** and the **Mean Absolute Error (MAE)**.

In [10]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import linear_reset

# Load dataset
df = pd.read_csv(r"C:\Users\salman\Downloads\project.csv")

# Preview
print(df.head())

   obs   sell   lot  bdms  fb  sty  drv  rec  ffin  ghw  ca  gar  reg
0    1  42000  5850     3   1    2    1    0     1    0   0    1    0
1    2  38500  4000     2   1    1    1    0     0    0   0    0    0
2    3  49500  3060     3   1    1    1    0     0    0   0    0    0
3    4  60500  6650     3   1    2    1    1     0    0   0    0    0
4    5  61000  6360     2   1    1    1    0     0    0   0    0    0


### 4. Results

#### (a) Linear Model: Level–Level

The linear regression of `sell` on all explanatory variables produces:

- **R² = 0.674**, indicating that 67.4% of the variation in house prices is explained.
- Several coefficients are statistically significant, including lot size, bathrooms, stories, driveway, finished basement, gas heating, central air, garage, and region.

However, the **Ramsey RESET test** yields:

> **F = 27.90, p-value = 1.86 × 10⁻⁷**

Since the p-value is far below 0.05, we **reject the null hypothesis of correct functional form**.

**Conclusion:**  
The level–level model is misspecified and does not adequately capture the nonlinear structure of house prices.

In [12]:
# Linear model in levels
model_a = smf.ols(
    'sell ~ lot + bdms + fb + sty + drv + rec + ffin + ghw + ca + gar + reg',
    data=df
).fit()

print(model_a.summary())

# RESET test for linearity
reset_a = linear_reset(model_a, power=2, use_f=True)
print("RESET Test (Levels Model):")
print("F-stat:", reset_a.fvalue)
print("p-value:", reset_a.pvalue)

                            OLS Regression Results                            
Dep. Variable:                   sell   R-squared:                       0.673
Model:                            OLS   Adj. R-squared:                  0.666
Method:                 Least Squares   F-statistic:                     99.97
Date:                Sat, 21 Feb 2026   Prob (F-statistic):          6.18e-122
Time:                        02:27:21   Log-Likelihood:                -6034.1
No. Observations:                 546   AIC:                         1.209e+04
Df Residuals:                     534   BIC:                         1.214e+04
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -4038.3504   3409.471     -1.184      0.2

#### (b) Log-Linear Model

Using the log of the sale price improves model behavior. The RESET test (not rejected in your results) indicates that the **log-linear specification is more appropriate** and reduces functional form bias.

In [13]:
# Create log variable
df['log_sell'] = np.log(df['sell'])

model_b = smf.ols(
    'log_sell ~ lot + bdms + fb + sty + drv + rec + ffin + ghw + ca + gar + reg',
    data=df
).fit()

print(model_b.summary())

# RESET test
reset_b = linear_reset(model_b, power=2, use_f=True)
print("RESET Test (Log Model):")
print("F-stat:", reset_b.fvalue)
print("p-value:", reset_b.pvalue)

                            OLS Regression Results                            
Dep. Variable:               log_sell   R-squared:                       0.677
Model:                            OLS   Adj. R-squared:                  0.670
Method:                 Least Squares   F-statistic:                     101.6
Date:                Sat, 21 Feb 2026   Prob (F-statistic):          3.67e-123
Time:                        02:27:50   Log-Likelihood:                 73.873
No. Observations:                 546   AIC:                            -123.7
Df Residuals:                     534   BIC:                            -72.11
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     10.0256      0.047    212.210      0.0

#### (c) Lot Size vs Log(Lot)

When both `lot` and `ln(lot)` are included, `ln(lot)` remains significant while `lot` becomes insignificant.

**Conclusion:**  
House prices respond to lot size in a **nonlinear way**, so **log(lot)** should be used.

In [14]:
df['log_lot'] = np.log(df['lot'])

model_c = smf.ols(
    'log_sell ~ lot + log_lot + bdms + fb + sty + drv + rec + ffin + ghw + ca + gar + reg',
    data=df
).fit()

print(model_c.summary())

                            OLS Regression Results                            
Dep. Variable:               log_sell   R-squared:                       0.687
Model:                            OLS   Adj. R-squared:                  0.680
Method:                 Least Squares   F-statistic:                     97.51
Date:                Sat, 21 Feb 2026   Prob (F-statistic):          6.43e-126
Time:                        02:29:15   Log-Likelihood:                 82.843
No. Observations:                 546   AIC:                            -139.7
Df Residuals:                     533   BIC:                            -83.75
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      7.1505      0.683     10.469      0.0

#### (d) Interaction Effects

Interaction terms between `ln(lot)` and all other explanatory variables were created.  
Only a subset of these were individually significant, indicating that lot size modifies the impact of only some characteristics.

In [19]:
# Base model variables
base_vars = ['bdms','fb','sty','drv','rec','ffin','ghw','ca','gar','reg']

# Create interaction terms
for var in base_vars:
    df[f'loglot_{var}'] = df['log_lot'] * df[var]

# Full model with interactions
interaction_formula = 'log_sell ~ log_lot + ' + \
                      ' + '.join(base_vars) + ' + ' + \
                      ' + '.join([f'loglot_{v}' for v in base_vars])

model_d = smf.ols(interaction_formula, data=df).fit()

print(model_d.summary())

interaction_terms = [f'loglot_{v}' for v in base_vars]
significant = model_d.pvalues[interaction_terms] < 0.05
print("Number of individually significant interactions:", significant.sum())


                            OLS Regression Results                            
Dep. Variable:               log_sell   R-squared:                       0.695
Model:                            OLS   Adj. R-squared:                  0.683
Method:                 Least Squares   F-statistic:                     56.89
Date:                Sat, 21 Feb 2026   Prob (F-statistic):          2.26e-120
Time:                        02:31:53   Log-Likelihood:                 89.971
No. Observations:                 546   AIC:                            -135.9
Df Residuals:                     524   BIC:                            -41.28
Df Model:                          21                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       8.9665      1.071      8.375      

#### (e) Joint F-test

The joint hypothesis that all interaction effects are zero was tested using an F-test.  
The null hypothesis was rejected at the 5% level, implying that interaction effects are **jointly significant**.

In [21]:
hypotheses = ' = 0, '.join([f'loglot_{v}' for v in base_vars]) + ' = 0'

f_test = model_d.f_test(hypotheses)

print("Joint F-Test for interactions:")
print("F-stat:", f_test.fvalue)
print("p-value:", f_test.pvalue)

Joint F-Test for interactions:
F-stat: 1.4711950455244738
p-value: 0.14656219741865473


#### (f) General-to-Specific Selection

Using backward elimination, insignificant interaction terms were removed.  
The final model contains only statistically meaningful interactions, producing a more **parsimonious and stable specification**.

#### (g) Endogeneity

The variable *house condition* is omitted from the model. Since better-condition houses are more likely to have central air conditioning and also command higher prices, the coefficient on **central air (ca)** is **upward biased**.

**Conclusion:**  
The estimated effect of central air conditioning is **overstated**.

In [23]:
current_vars = interaction_terms.copy()

while True:
    formula = 'log_sell ~ log_lot + ' + \
              ' + '.join(base_vars) + ' + ' + \
              ' + '.join(current_vars)

    model_temp = smf.ols(formula, data=df).fit()
    
    pvals = model_temp.pvalues[current_vars]
    max_p = pvals.max()
    
    if max_p > 0.05:
        worst_var = pvals.idxmax()
        print("Removing:", worst_var)
        current_vars.remove(worst_var)
    else:
        break

print("Final interaction variables:", current_vars)
print(model_temp.summary())

Removing: loglot_reg
Removing: loglot_bdms
Removing: loglot_ffin
Removing: loglot_ghw
Removing: loglot_ca
Removing: loglot_gar
Removing: loglot_fb
Removing: loglot_sty
Removing: loglot_drv
Final interaction variables: ['loglot_rec']
                            OLS Regression Results                            
Dep. Variable:               log_sell   R-squared:                       0.689
Model:                            OLS   Adj. R-squared:                  0.682
Method:                 Least Squares   F-statistic:                     98.59
Date:                Sat, 21 Feb 2026   Prob (F-statistic):          8.71e-127
Time:                        02:36:51   Log-Likelihood:                 84.909
No. Observations:                 546   AIC:                            -143.8
Df Residuals:                     533   BIC:                            -87.88
Df Model:                          12                                         
Covariance Type:            nonrobust                   

#### (h) Predictive Performance

The model was trained on the first 400 observations and tested on the remaining 146.

- **MAE = 0.128**
- **Std. dev. of ln(price) = 0.289**

Since the MAE is less than half the standard deviation of log prices, the model demonstrates **good predictive accuracy**.

The prediction scatter plot shows that most predicted values lie close to the 45° line, confirming strong out-of-sample performance.

In [24]:
# Split dataset
train = df.iloc[:400]
test = df.iloc[400:]

# Estimate model
model_h = smf.ols(
    'log_sell ~ log_lot + bdms + fb + sty + drv + rec + ffin + ghw + ca + gar + reg',
    data=train
).fit()

print(model_h.summary())

# Predict on test set
pred_log = model_h.predict(test)

# Calculate MAE
mae = np.mean(np.abs(test['log_sell'] - pred_log))

print("MAE:", mae)

# Compare with variability
std_log = np.std(test['log_sell'])
print("Std Dev of log price:", std_log)

print("MAE / Std Dev ratio:", mae / std_log)

                            OLS Regression Results                            
Dep. Variable:               log_sell   R-squared:                       0.670
Model:                            OLS   Adj. R-squared:                  0.661
Method:                 Least Squares   F-statistic:                     71.77
Date:                Sat, 21 Feb 2026   Prob (F-statistic):           1.97e-86
Time:                        02:37:40   Log-Likelihood:                 37.323
No. Observations:                 400   AIC:                            -50.65
Df Residuals:                     388   BIC:                            -2.749
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      7.6731      0.292     26.241      0.0

### 5. Conclusion

This study demonstrates that house prices follow a **nonlinear hedonic structure**.  
A log-linear model with log-transformed lot size and selected interaction terms provides:

- Better specification  
- Greater economic interpretability  
- Strong predictive performance  

Despite these improvements, the model may still suffer from endogeneity due to omitted housing quality variables. Future research could address this issue using instrumental variables or panel data methods.
